In [7]:
import pandas as pd

In [8]:
# Load all datasets
census = pd.read_csv("census_wide.csv")
zillow = pd.read_csv("ZillowHousingData.csv")
density = pd.read_csv("business_density_wide.csv")
crosswalk = pd.read_csv("zip_tract_crosswalk.csv")

Merging Census and Density data (LODES)

In [14]:
from google.colab import files

print("Upload tract_areas.csv")
uploaded = files.upload()

gaz = pd.read_csv("tract_areas.csv", dtype={"tract_id": str})
gaz["tract_id"] = gaz["tract_id"].str.zfill(11)
print(f"Tract areas loaded: {len(gaz):,}")
print(gaz.head(3))

Upload tract_areas.csv


Saving tract_areas.csv to tract_areas (1).csv
Tract areas loaded: 1,296
      tract_id  area_sqmi
0  06085509201   0.609718
1  06085510500   1.000304
2  06085509401   0.257310


In [17]:
import numpy as np

def normalize_id(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(11)

census = pd.read_csv("census_wide.csv")
census["tract_id"] = normalize_id(census["tract_id"])

lodes = pd.read_csv("business_density_wide.csv")
lodes["tract_id"] = normalize_id(lodes["tract_id"])

df = census.merge(lodes, on="tract_id", how="left")

# Merge gazetteer areas and rename column before backfill
df = df.merge(gaz, on="tract_id", how="left")
df = df.rename(columns={"area_sqmi": "area_sqmi_census"})

for year in [2010, 2015, 2020, 2022]:
    col = f"area_sqmi_{year}"
    if col in df.columns:
        missing = df[col].isna()
        df.loc[missing, col] = df.loc[missing, "area_sqmi_census"]
        df[f"arts_density_{year}"]     = df[f"jobs_arts_{year}"] / df[col]
        df[f"food_density_{year}"]     = df[f"jobs_food_{year}"] / df[col]
        df[f"gentrify_density_{year}"] = (df[f"jobs_arts_{year}"] + df[f"jobs_food_{year}"]) / df[col]

df = df.drop(columns=["area_sqmi_census"])

print(f"Merged shape: {df.shape[0]:,} tracts × {df.shape[1]} columns")
print(f"Tracts with job data:       {df['jobs_arts_2010'].notna().sum():,}")
print(f"Tracts with valid density:  {df['arts_density_2010'].notna().sum():,}")
print(f"Remaining NaN density:      {df['arts_density_2010'].isna().sum():,}")

df.to_csv("census_and_LODES_data_wide.csv", index=False)
print("\nSaved: census_and_LODES_data_wide.csv")
df.head(3)

Merged shape: 1,590 tracts × 79 columns
Tracts with job data:       1,440
Tracts with valid density:  1,149
Remaining NaN density:      441

Saved: census_and_LODES_data_wide.csv


,tract_id,population_2010,median_income_2010,median_rent_2010,median_home_value_2010,share_college_2010,share_25_34_2010,homeownership_rate_2010,vacancy_rate_2010,share_pre1940_2010,...,area_sqmi_2020,jobs_arts_2022,jobs_food_2022,jobs_total_2022,arts_density_2022,food_density_2022,gentrify_density_2022,arts_job_share_2022,food_job_share_2022,area_sqmi_2022
0,06001400100,2701.0,173472.0,2001.0,1000001.0,0.841828,0.021103,0.883371,0.078363,0.090846,...,2.657886,0.0,17.0,285.0,0.000000,6.396060,6.396060,0.000000,0.059649,2.657886
1,06001400200,2050.0,95833.0,1342.0,902400.0,0.810191,0.085854,0.683921,0.036093,0.801486,...,0.229930,0.0,232.0,1156.0,0.000000,1009.004334,1009.004334,0.000000,0.200692,0.229930
2,06001400300,4719.0,52314.0,998.0,740000.0,0.674725,0.093028,0.424652,0.069552,0.616352,...,0.426612,15.0,677.0,2022.0,35.160786,1586.923454,1622.084240,0.007418,0.334817,0.426612


In [18]:
# Download the feature_matrix_wide.csv as Census_and_LODES_data.csv
files.download('census_and_LODES_data_wide.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Merge Zillow Data with census_and_LODES_data